# QuetHoaDon · Giai đoạn 1: nâng độ chính xác OCR (VietOCR tinh chỉnh + lọc hộp chồng nhau)

Chạy trên Kaggle: **GPU T4**, Internet bật. Gắn 3 dataset như notebook 05: `m1b-v2-rs` (checkpoint M1b Tuần 4), `quethoadon` (token dataset v1, ảnh đã tiền xử lý), **Vietnamese Receipts MC_OCR 2021** (ảnh gốc + CSV nhãn).

Mọi quyết định chỉ dựa trên **validation**; tập test không được đọc. Thứ tự:
1. Chạy lại OCR gốc (VietOCR chưa tinh chỉnh) → kiểm tra tái lập token dataset v1.
2. Tạo cặp (ảnh cắt, chữ đúng) từ tập train → tinh chỉnh VietOCR → so CER với bản gốc trên validation (**cổng 1**).
3. Chạy OCR toàn bộ với VietOCR đã tinh chỉnh, thử thêm bộ lọc hộp chồng nhau → so "trần trên" của từng bản OCR (**cổng 2**).
4. Huấn luyện lại M1b trên bản OCR tốt nhất → chấm "đọc chuẩn" trên validation, so với mốc 52/14/75/40/72/10.

Tổng thời gian GPU ước tính 3–4 giờ. Dừng ở cell lỗi và gửi output. Cell `[KIỂM THỬ]` ghi rõ kết quả mong đợi.

In [ ]:
# CELL 1 - Cài detectron2 (LayoutXLM cần, lần đầu 5-10 phút). Mong đợi: "detectron2 OK"
try:
    import detectron2
    print("detectron2 đã có:", detectron2.__version__)
except ImportError:
    !pip install -q 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation 2>&1 | tail -3
    import detectron2; print("detectron2 OK", detectron2.__version__)

In [ ]:
# CELL 2 - Lấy mã nguồn nhánh HoaiNam/ocr-finetune và cài thư viện OCR. Mong đợi: một dòng commit, không lỗi pip
BRANCH = "HoaiNam/ocr-finetune"
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH} && git log --oneline -1
!pip install -q -r requirements-dev.txt
!pip install -q craft-text-detector vietocr --no-deps
!pip install -q gdown einops prefetch_generator albumentations lmdb 2>&1 | tail -2
print("xong")

In [ ]:
# CELL 3 [KIỂM THỬ] - GPU, dữ liệu, checkpoint Tuần 4 và unit test.
# Mong đợi: "GPU: True Tesla T4"; số ảnh gốc khoảng 1155; CKPT thuộc run m1b_v2_20261110_183118; "Ran ... tests" rồi "OK"
import glob, os, sys, json, torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
sys.path.insert(0, '/kaggle/working/QuetHoaDon')
TOK = os.path.dirname(glob.glob('/kaggle/input/**/token_dataset_v1*/**/validation.jsonl', recursive=True)[0])
CSV = glob.glob('/kaggle/input/**/mcocr_train_df.csv', recursive=True)[0]
IMG_RAW = os.path.dirname(glob.glob('/kaggle/input/**/train_images/**/*.jpg', recursive=True)[0])
found = sorted(glob.glob('/kaggle/input/**/runs/m1b_v2_*/best', recursive=True))
CKPT = found[-1] if found else None
print("TOK =", TOK); print("CSV =", CSV); print("IMG_RAW =", IMG_RAW, "| số ảnh gốc:", len(os.listdir(IMG_RAW)))
print("CKPT =", CKPT)
assert CKPT and os.path.basename(os.path.dirname(CKPT)) == "m1b_v2_20261110_183118", "checkpoint không phải run Tuần 4"
W = '/kaggle/working'
os.makedirs(f'{W}/eval', exist_ok=True)
!python -m unittest discover -s tests 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"
# Nếu có test lỗi: chạy lại với "2>&1 | tail -40" và gửi phần lỗi.

In [ ]:
# CELL 4 [KIỂM THỬ] - Tạo file chia tập và kiểm tra trùng với split đã dùng để train ở Tuần 4.
# Mong đợi: "validation khớp token dataset: True (115 hóa đơn ...)". Nếu False thì DỪNG và gửi output.
import csv
from src.training.data import load_jsonl
!python scripts/prepare_dataset.py --csv "{CSV}" --images "{IMG_RAW}" --out {W}/dataset > {W}/dataset_summary.txt
MANIFEST = f'{W}/dataset/mcocr_manifest_split.csv'
with open(MANIFEST, encoding='utf-8') as f:
    split_rows = list(csv.DictReader(f))
for sp in ("train", "validation"):
    new = {r['img_id'] for r in split_rows if r['split'] == sp}
    old = {r['img_id'] for r in load_jsonl(os.path.join(TOK, f'{sp}.jsonl'))}
    print(f"{sp} khớp token dataset:", new >= old, f"({len(new)} hóa đơn; token dataset {len(old)})")

In [ ]:
# CELL 5 [KIỂM THỬ] - Chạy thử OCR trên 5 ảnh (1-2 phút, tải trọng số CRAFT/VietOCR lần đầu).
# Mong đợi: dòng "[done] {'ok': 5, ...}" và vài dòng chữ tiếng Việt đọc được
!python -u scripts/run_ocr_dataset.py --images "{IMG_RAW}" --split-manifest {MANIFEST} --out {W}/ocr_smoke --limit 5
j = json.load(open(sorted(glob.glob(f'{W}/ocr_smoke/ocr_annotations/*.json'))[0]))
print(j['file_id'], '| số vùng:', j['n_text_regions']); print([it['text'] for it in j['ocr_items'][:12]])

## 1. Chạy lại OCR gốc và kiểm tra tái lập

In [ ]:
# CELL 6 - OCR gốc (VietOCR chưa tinh chỉnh) trên toàn bộ ảnh, khoảng 30-60 phút. Mong đợi: "[done] {'ok': ...}" và failed gần 0
# Nếu session ngắt: chạy lại cell 1-4 rồi cell này, ảnh đã xong sẽ được bỏ qua.
OCR_V1R = f'{W}/ocr_v1r'
!python -u scripts/run_ocr_dataset.py --images "{IMG_RAW}" --split-manifest {MANIFEST} --out {OCR_V1R}

In [ ]:
# CELL 7 [KIỂM THỬ] - Dựng token dataset từ OCR vừa chạy và so với token dataset v1 (CPU, khoảng 1 phút).
# Mong đợi: "identical words in X/115" với X gần 115. Nếu X nhỏ hơn nhiều (< 100): GỬI output, môi trường đọc khác lần đầu
TOK_V1R = f'{W}/tok_v1r'
!python scripts/build_token_dataset.py --csv "{CSV}" --ocr-dir {OCR_V1R}/ocr_annotations --split-manifest {MANIFEST} --out {TOK_V1R} | head -5
!python scripts/compare_ocr_runs.py --reference v1="{TOK}" --run v1r={TOK_V1R} --out {W}/eval/ocr_runs_v1r_val.json

## 2. Tinh chỉnh VietOCR (cổng 1)

In [ ]:
# CELL 8 [KIỂM THỬ] - Tạo cặp (ảnh cắt, chữ đúng) cho train và validation từ OCR gốc (CPU, vài phút).
# Mong đợi: train có vài nghìn "line" và "region"; out_of_vocab, frame_mismatch nhỏ
PAIRS = f'{W}/ocr_pairs'
!python scripts/finetune_vietocr.py build --csv "{CSV}" --ocr-dir {OCR_V1R}/ocr_annotations --image-dir {OCR_V1R}/preprocessed_images --split-manifest {MANIFEST} --out {PAIRS}

In [ ]:
# CELL 9 [KIỂM THỬ] - Chạy thử vòng huấn luyện với 256 cặp, 1 epoch (2-4 phút).
# Mong đợi: dòng "[ft] epoch 0 (pretrained) ...", "[ft] epoch 1 ..." và bảng CER; không lỗi
!python -u scripts/finetune_vietocr.py train --pairs {PAIRS} --out {W}/vietocr_smoke --epochs 1 --max-train 256

In [ ]:
# CELL 10 - Tinh chỉnh đầy đủ (khoảng 20-60 phút; in tiến độ ~20% mỗi epoch). CỔNG 1: gửi bảng cuối cho mình.
# Mong đợi: CER của "kind region" sau tinh chỉnh thấp hơn rõ bản gốc, khoảng tin cậy của "region" nằm hẳn dưới 0
VFT = f'{W}/vietocr_ft'
!python -u scripts/finetune_vietocr.py train --pairs {PAIRS} --out {VFT} --epochs 12 --lr 1e-4

## 3. OCR với VietOCR đã tinh chỉnh và bộ lọc hộp (cổng 2)
Chỉ chạy tiếp khi cổng 1 đạt (CER region giảm rõ). Nếu không đạt, gửi output và dừng.

In [ ]:
# CELL 11 - OCR toàn bộ ảnh với VietOCR đã tinh chỉnh (khoảng 30-60 phút). Mong đợi: "[done] ..." với weights=.../vietocr_finetuned.pth
OCR_FT = f'{W}/ocr_ft'
!python -u scripts/run_ocr_dataset.py --images "{IMG_RAW}" --split-manifest {MANIFEST} --out {OCR_FT} --weights {VFT}/vietocr_finetuned.pth

In [ ]:
# CELL 12 [KIỂM THỬ] - Thêm biến thể lọc hộp chồng nhau (ngưỡng 0.5 thay vì 0.8), dựng token dataset và so "trần trên" (CPU, 2-3 phút).
# Mong đợi: bảng 5 dòng (v1, v1r, v1r_n05, ft, ft_n05); CHOSEN được chọn tự động theo luật viết sẵn bên dưới
for name, src in (("v1r_n05", OCR_V1R), ("ft_n05", OCR_FT)):
    dst = f"{W}/ocr_{name}/ocr_annotations"
    !python scripts/filter_nested_ocr.py --in-dir {src}/ocr_annotations --out-dir {dst} --threshold 0.5
TOKS = {"v1r": TOK_V1R, "v1r_n05": f"{W}/tok_v1r_n05", "ft": f"{W}/tok_ft", "ft_n05": f"{W}/tok_ft_n05"}
OCRS = {"v1r_n05": f"{W}/ocr_v1r_n05", "ft": OCR_FT, "ft_n05": f"{W}/ocr_ft_n05"}
for name, ocr in OCRS.items():
    tok_out = TOKS[name]
    !python scripts/build_token_dataset.py --csv "{CSV}" --ocr-dir {ocr}/ocr_annotations --split-manifest {MANIFEST} --out {tok_out} > /dev/null
runs = " ".join(f"--run {k}={v}" for k, v in TOKS.items())
!python scripts/compare_ocr_runs.py --reference v1="{TOK}" {runs} --out {W}/eval/ocr_runs_val.json
# Luật chọn (viết trước khi xem kết quả): số khớp tuyệt đối SELLER + ADDRESS nhiều nhất trên validation (TOTAL và TIMESTAMP không so được vì chữ nhãn có từ khóa và giờ), hòa thì CER SELLER + ADDRESS thấp hơn
o = json.load(open(f'{W}/eval/ocr_runs_val.json'))['oracle']
score = {k: (sum(v[f]['exact'] for f in ('SELLER', 'ADDRESS')), -sum(v[f]['mean_cer'] for f in ('SELLER', 'ADDRESS'))) for k, v in o.items() if k != 'v1'}
CHOSEN = max(score, key=score.get)
TOKC = TOKS[CHOSEN]
print("điểm (khớp tuyệt đối, -CER):", score); print("CHOSEN =", CHOSEN, "| TOKC =", TOKC)

## 4. Huấn luyện lại M1b trên OCR đã chọn và chấm "đọc chuẩn"

In [ ]:
# CELL 13 - Huấn luyện M1b (cùng cấu hình M1b v2, seed 42) trên token dataset CHOSEN (khoảng 25 phút).
# Mong đợi: device=cuda, các dòng [epoch n], cuối cùng JSON status early_stopped / finished
IMG_PRE = f'{OCR_V1R}/preprocessed_images'     # ảnh tiền xử lý giống nhau ở mọi bản OCR (cùng bước tiền xử lý)
!python -u scripts/train_layoutxlm.py --token-dir {TOKC} --image-dir {IMG_PRE} --out {W}/runs --name m1b_v3_{CHOSEN} --seed 42 --class-weight sqrt_inverse --select-metric macro --epochs 20 --patience 5
CK3 = sorted(glob.glob(f'{W}/runs/m1b_v3_{CHOSEN}_*/best'))[-1]; print("CK3 =", CK3)

In [ ]:
# CELL 14 [KIỂM THỬ] - Chấm validation (lưu xác suất) và giải mã kèm luật ngày giờ. Gửi mình bảng "Strict reading accuracy".
# So với mốc (M1b Tuần 4 + luật ngày giờ): SELLER 52/106, ADDRESS 14/112, ngày 75/99, giờ 40/80, tổng 72/98, cả HĐ 10/115
!python scripts/evaluate_layoutxlm.py --checkpoint "{CK3}" --token-dir {TOKC} --image-dir {IMG_PRE} --split validation --save-probs {W}/eval/m1b_v3_val_probs.npz --out {W}/eval/m1b_v3_val.json 2>&1 | tail -4
!python scripts/decode_from_probs.py --token-dir {TOKC} --split validation --probs {W}/eval/m1b_v3_val_probs.npz --datetime --out {W}/eval/decoded_v3_val.json

In [ ]:
# CELL 15 - Nén kết quả để tải về (tab Output): eval, báo cáo tinh chỉnh, trọng số VietOCR đã tinh chỉnh (~150 MB), token dataset mới
!cd /kaggle/working && zip -q -r phase1_eval.zip eval ocr_pairs/pairs_report.json vietocr_ft runs/*/run.json tok_* && ls -la phase1_eval.zip